In [254]:
import os
import copy
import torch
import numpy as np
import torch.nn as nn
from tqdm import tqdm
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from typing import Any
import math
from sklearn.preprocessing import StandardScaler

In [255]:
class ToTensor:
    def __call__(self, sample):
        return torch.tensor(sample, dtype=torch.float32)

In [256]:
class Normalize:
    def __call__(self, sample):
        mean = sample.mean(dim=1, keepdim=True)
        std = sample.std(dim=1, keepdim=True) + 1e-6
        return (sample - mean) / std

In [257]:
class GaussianNoise:
    def __init__(self, mean=0., std=1.):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        return tensor + torch.randn(tensor.size()) * self.std + self.mean

In [258]:
# class train_DatasetWrapper(Dataset):
#     def __init__(self, train_data_path, label_path):
#         self.X = np.load(train_data_path)   # shape: (samples, time) or (samples, 1, time)
#         self.y = np.load(label_path)        # shape: (samples,)

#         # if self.X.ndim == 2:
#         #     self.X = self.X[:, np.newaxis, :]  # reshape to (samples, 1, time)

#         self.to_tensor = ToTensor()
#         self.normalize = Normalize()
#         self.add_noise = GaussianNoise(mean=0.0, std=0.2)

#     def __len__(self):
#         return len(self.y)
    
#     def __getitem__(self, idx):
#         sample = self.X[idx]  # shape: (time,)
#         label = self.y[idx]

#         sample = self.to_tensor(sample)

#         if sample.dim() == 1:
#             sample = sample.unsqueeze(0)  # [1, time]

#         sample = sample
#         # sample = self.add_noise(sample) # Apply noise here for training data

#         label = torch.tensor(label, dtype=torch.float32)
#         return sample.float(), label

In [259]:
class train_DatasetWrapper(Dataset):
    def __init__(self, train_data_path, label_path):
        self.X = np.load(train_data_path)  # shape: (samples, 12, time)
        self.y = np.load(label_path)       # shape: (samples,)

        # Assertion to ensure the data has 12 channels
        assert self.X.shape[1] == 12, f"Expected 12 channels, but got {self.X.shape[1]}"

        self.to_tensor = ToTensor()
        self.normalize = Normalize()
        self.add_noise = GaussianNoise(mean=0.0, std=0.2)

    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        sample = self.X[idx]  # shape: (12, time)
        label = self.y[idx]

        sample = self.to_tensor(sample)
        sample = sample

        # Changed to torch.long for CrossEntropyLoss
        label = torch.tensor(int(label), dtype=torch.long)
        return sample.float(), label

In [260]:
# import numpy as np
# import torch
# from torch.utils.data import Dataset

# class train_DatasetWrapper(Dataset):
#     def __init__(self, train_data_path, label_path):
#         # Load data
#         self.X = np.load(train_data_path)  # shape: (samples, 12, time)
#         self.y = np.load(label_path, allow_pickle=True)

#         # === Fix labels: array([1]) → 1 ===
#         # Works for: array([1]), [[1]], or scalar
#         self.y = np.array([int(label) for label in self.y], dtype=np.int64)

#         # Ensure labels are 1D
#         assert self.y.ndim == 1, f"Labels must be 1D, got shape {self.y.shape}"

#         # Ensure correct channel count
#         assert self.X.shape[1] == 12, f"Expected 12 channels, got {self.X.shape[1]}"

#         self.to_tensor = ToTensor()
#         self.normalize = Normalize()
#         self.add_noise = GaussianNoise(mean=0.0, std=0.2)

#     def __len__(self):
#         return len(self.y)

#     def __getitem__(self, idx):
#         sample = self.X[idx]  # (12, time)
#         label = self.y[idx]   # scalar int

#         sample = self.to_tensor(sample)

#         # CrossEntropyLoss requires torch.long targets
#         label = torch.tensor(label, dtype=torch.long)

#         return sample.float(), label


In [261]:
clamp_val = 2.0

In [262]:
class LinearEmbed(nn.Module):
    def __init__(self, num_lead: int, chunk_len: int, d_model: int) -> None:
        super(LinearEmbed, self).__init__()

        self.num_lead = 12  # Changed to 12
        self.chunk_len = chunk_len

        chunk_dim = self.num_lead * self.chunk_len
        self.embed = nn.Linear(chunk_dim, d_model)

    def forward(self, x: torch.Tensor):
        assert(x.size(1) == self.num_lead)
        assert(x.size(2) % self.chunk_len == 0)

        bs = x.size(0)
        num_chunks = x.size(2) // self.chunk_len
        x = torch.reshape(x, (bs, self.num_lead, num_chunks, self.chunk_len))
        x = x.permute(0, 2, 1, 3)

        x = torch.reshape(x, (bs, num_chunks, -1))

        feat = self.embed(x)
        return feat

In [263]:
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=3):
        super(SpatialAttention, self).__init__()
        self.conv1 = nn.Conv1d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (batch, channels, seq_len)
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        concat = torch.cat([avg_out, max_out], dim=1)  # (batch, 2, seq_len)
        attn = self.sigmoid(self.conv1(concat))        # (batch, 1, seq_len)
        return x * attn  

In [264]:
class ConvEmbed(nn.Module):
    def __init__(self, num_lead: int, d_model: int) -> None:
        super(ConvEmbed, self).__init__()

        self.num_lead = num_lead

        self.conv1 = nn.Conv1d(num_lead, 128, kernel_size=14, stride=3, padding=2)
        # self.attn1 = SpatialAttention()

        self.conv2 = nn.Conv1d(128, 256, kernel_size=14, stride=3, padding=0)
        # self.attn2 = SpatialAttention()

        self.conv3 = nn.Conv1d(256, 256, kernel_size=10, stride=2, padding=0)
        # self.attn3 = SpatialAttention()

        # self.conv4 = nn.Conv1d(256, 256, kernel_size=10, stride=2, padding=0)
        # self.attn4 = SpatialAttention()

        # self.conv5 = nn.Conv1d(256, 256, kernel_size=10, stride=1, padding=0)
        # self.attn5 = SpatialAttention()

        self.conv6 = nn.Conv1d(256, 256, kernel_size=10, stride=1, padding=0)
        self.attn6 = SpatialAttention()

        self.dense = nn.Linear(256, d_model)

    def forward(self, x: torch.Tensor):
        """
        Args:
            x (torch.Tensor): (batch_size, num_lead, seqlen)
        Returns:
            feat (torch.Tensor): (batch_size, num_steps, d_model)
        """
        feat = self.conv1(x)
        # feat = self.attn1(feat)

        feat = self.conv2(feat)
        # feat = self.attn2(feat)

        feat = self.conv3(feat)
        # feat = self.attn3(feat)

        # feat = self.conv4(feat)
        # feat = self.attn4(feat)

        # feat = self.conv5(feat)
        # feat = self.attn5(feat)

        feat = self.conv6(feat)
        feat = self.attn6(feat)

        feat = feat.permute(0, 2, 1)  # (B, T, C)
        feat = self.dense(feat)       # (B, T, d_model)

        return feat


In [265]:
class PositionalEncoding(nn.Module):
    """
    From `https://pytorch.org/tutorials/beginner/transformer_tutorial.html`
    """

    def __init__(
        self,
        d_model: int,
        dropout: float=0.3,
        max_len: int=5000
    ) -> None:
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor):
        """
        Add positional encoding.
        Args:
            x (torch.Tensor): Tensor of size (batch_size, num_steps, d_model).
        Returns:
            x (torch.Tensor): Tensor of size (batch_size, num_steps, d_model)
        """
        seq_len = x.size(1)
        x = x + self.pe[:seq_len, :].unsqueeze(0)
        return self.dropout(x)

In [266]:
import math
from enum import Enum
import torch
import torch.nn as nn


class MixingMatrixInit(Enum):
    CONCATENATE = 1
    ALL_ONES = 2
    UNIFORM = 3


class CollaborativeAttention(nn.Module):
    def __init__(
        self,
        dim_input: int,
        dim_value_all: int,
        dim_key_query_all: int,
        dim_output: int,
        num_attention_heads: int,
        output_attentions: bool,
        attention_probs_dropout_prob: float,
        use_dense_layer: bool,
        use_layer_norm: bool,
        mixing_initialization: MixingMatrixInit = MixingMatrixInit.UNIFORM,
    ):
        super().__init__()

        if dim_value_all % num_attention_heads != 0:
            raise ValueError(
                "Value dimension ({}) should be divisible by number of heads ({})".format(
                    dim_value_all, num_attention_heads
                )
            )

        if not use_dense_layer and dim_value_all != dim_output:
            raise ValueError(
                "Output dimension ({}) should be equal to value dimension ({}) if no dense layer is used".format(
                    dim_output, dim_value_all
                )
            )

        # save args
        self.dim_input = dim_input
        self.dim_value_all = dim_value_all
        self.dim_key_query_all = dim_key_query_all
        self.dim_output = dim_output
        self.num_attention_heads = num_attention_heads
        self.output_attentions = output_attentions
        self.attention_probs_dropout_prob = attention_probs_dropout_prob
        self.mixing_initialization = mixing_initialization
        self.use_dense_layer = use_dense_layer
        self.use_layer_norm = use_layer_norm

        self.dim_value_per_head = dim_value_all // num_attention_heads
        self.attention_head_size = (
            dim_key_query_all / num_attention_heads
        )  # does not have to be integer

        # intialize parameters
        self.query = nn.Linear(dim_input, dim_key_query_all, bias=False)
        self.key = nn.Linear(dim_input, dim_key_query_all, bias=False)
        self.content_bias = nn.Linear(dim_input, num_attention_heads, bias=False)
        self.value = nn.Linear(dim_input, dim_value_all)

        # del self.m_t
        # del self.m_c
        self.m_t = self.init_mixing_matrix()
        self.m_c = self.init_mixing_matrix()

        self.dense = (
            nn.Linear(dim_value_all, dim_output) if use_dense_layer else nn.Sequential()
        )

        self.dropout = nn.Dropout(attention_probs_dropout_prob)

        if use_layer_norm:
            self.layer_norm = nn.LayerNorm(dim_value_all)

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        head_mask=None,
        encoder_hidden_states=None,
        encoder_attention_mask=None,
    ):
        # import ipdb; ipdb.set_trace()
        from_sequence = hidden_states
        to_sequence = hidden_states
        # print(from_sequence.shape)
        # print(to_sequence.shape)
        if encoder_hidden_states is not None:
            to_sequence = encoder_hidden_states
            attention_mask = encoder_attention_mask
            # print(attention_mask.shape)

        query_layer = self.query(from_sequence)
        key_layer = self.key(to_sequence)
        # print(f"query_layer shape {query_layer.shape}")
        # print(f"Key_layer shape {key_layer.shape}")

        # point wise multiplication of the mixing coefficient per head with the shared query projection
        # (batch, from_seq, dim) x (head, dim) -> (batch, head, from_seq, dim)
        mixed_query = query_layer[..., None, :, :] * self.m_c[..., :, None, :]
        # print(f"self mixing {self.mixing.shape}")
        # print(f"mixed_query {mixed_query.shape}")

        # broadcast the shared key for all the heads
        # (batch, 1, to_seq, dim)
        mixed_key = key_layer[..., None, :, :]* self.m_t[..., :, None, :]
        # print(f"mixed key {mixed_key.shape}")

        # (batch, head, from_seq, to_seq)
        attention_scores = torch.matmul(mixed_query, mixed_key.transpose(-1, -2))
        # print(f"attention_score {attention_scores.shape}")

        # add the content bias term
        # (batch, to_seq, heads)
        content_bias = self.content_bias(to_sequence)
        # (batch, heads, 1, to_seq)
        # print(f'content_bias {content_bias.shape}')
        broadcast_content_bias = content_bias.transpose(-1, -2).unsqueeze(-2)
        # print(f"broadcast_content_bias {broadcast_content_bias.shape}")
        attention_scores += broadcast_content_bias
        # print(f"attention_scores: {attention_scores.shape}")

        attention_scores = attention_scores / math.sqrt(self.attention_head_size)
        if attention_mask is not None:
            attention_scores = attention_scores + attention_mask
            # print(f"attention_scores if mask: {attention_scores.shape}")

        # Normalize the attention scores to probabilities.
        attention_probs = nn.Softmax(dim=-1)(attention_scores)
        # print(f"attention probs: {attention_probs.shape}")

        # This is actually dropping out entire tokens to attend to, which might
        # seem a bit unusual, but is taken from the original Transformer paper.
        attention_probs = self.dropout(attention_probs)

        # Mask heads if we want to
        if head_mask is not None:
            attention_probs = attention_probs * head_mask

        value_layer = self.value(to_sequence)
        value_layer = self.transpose_for_scores(value_layer)
        context_layer = torch.matmul(attention_probs, value_layer)
        # print(f"value layer {value_layer.shape}")
        # print(f"context_layer {context_layer.shape}")

        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        # print(f"context_layer after permute {context_layer.shape}")
        new_context_layer_shape = context_layer.size()[:-2] + (self.dim_value_all,)
        # print(f"new_context_layer_shape after permute {new_context_layer_shape}")
        context_layer = context_layer.view(*new_context_layer_shape)
        # print(f"context_layer after permute {context_layer.size}")

        context_layer = self.dense(context_layer)
        # print(f"context_layer after passing in dense {context_layer.shape}")

        if self.use_layer_norm:
            context_layer = self.layer_norm(from_sequence + context_layer)

        if self.output_attentions:
            return (context_layer, attention_probs)
        else:
            return (context_layer,)

    def transpose_for_scores(self, x):
        new_x_shape = x.size()[:-1] + (self.num_attention_heads, -1)
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)

    def init_mixing_matrix(self, scale=0.2):
        mixing = torch.zeros(self.num_attention_heads, self.dim_key_query_all)

        if self.mixing_initialization is MixingMatrixInit.CONCATENATE:
            # last head will be smaller if not equally divisible
            dim_head = int(math.ceil(self.dim_key_query_all / self.num_attention_heads))
            for i in range(self.num_attention_heads):
                mixing[i, i * dim_head : (i + 1) * dim_head] = 1.0
                # print(f"mixing shape {mixing.shape}")

        elif self.mixing_initialization is MixingMatrixInit.ALL_ONES:
            mixing.fill_(1.0)
            
        elif self.mixing_initialization is MixingMatrixInit.UNIFORM:
            mixing.normal_(std=scale)
        else:
            raise ValueError(
                "Unknown mixing matrix initialization: {}".format(
                    self.mixing_initialization
                )
            )

        return nn.Parameter(mixing)

In [267]:
class TransformerEncoderLayerWithCollaborativeAttention(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = CollaborativeAttention(
            dim_input=d_model, dim_value_all=d_model, dim_key_query_all=d_model,
            dim_output=d_model, num_attention_heads=num_heads,
            output_attentions=False, attention_probs_dropout_prob=dropout,
            use_dense_layer=True, use_layer_norm=False,  # <-- off here
            mixing_initialization=MixingMatrixInit.UNIFORM,
        )
        self.dropout1 = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.GELU(),                    # GELU works nicely on ECG
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model)
        )
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        # Pre-LN
        h = x
        x = self.norm1(x)
        x = self.attn(x)[0]
        x = h + self.dropout1(x)

        h = x
        x = self.norm2(x)
        x = self.ff(x)
        # x = h + self.dropout2(x)
        return x


In [268]:
class TransformerModel(nn.Module):
    def __init__(self, num_layers: int, num_heads: int, d_model: int, ff_dim: int, embed_mode: str, num_lead: int = 12, chunk_len: int = 50, embedding_dim: int = 512, **kwargs,) -> None:
        super(TransformerModel, self).__init__()
        self.embedding_dim = embedding_dim
        print("Transformer with embedding dim {}".format(self.embedding_dim))

        if embed_mode == "linear":
            self.embed = LinearEmbed(num_lead, chunk_len, d_model)
        elif embed_mode == "cnn":
            self.embed = ConvEmbed(num_lead, d_model)
        else:
            raise NotImplementedError(f"Embedding model `{embed_mode}` is not implemented")

        self.pos_encoder = PositionalEncoding(d_model)

        # Replace with custom encoder layer
        self.transformer_encoder = nn.Sequential(
            *[TransformerEncoderLayerWithCollaborativeAttention(d_model, num_heads, ff_dim) for _ in range(num_layers)]
        )

        self.fc = nn.Linear(d_model, self.embedding_dim)
        self.name = "transformer"

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        
        feat = self.embed(x)
        feat = torch.clamp(feat, min=-1*clamp_val, max=clamp_val)
        feat = self.pos_encoder(feat)
        feat = self.transformer_encoder(feat)

        feat = feat.permute(0, 2, 1)
        feat = nn.AdaptiveAvgPool1d(1)(feat)
        feat = feat.squeeze(-1)
        feat = self.fc(feat)
        return feat


In [269]:
def _transformer(
    arch: str,
    num_layers: int,
    num_heads: int,
    d_model: int,
    ff_dim: int,
    embed_mode: str,
    **kwargs: Any,
) -> TransformerModel:
    model = TransformerModel(
        num_layers, num_heads, d_model, ff_dim, embed_mode, **kwargs)
    return model

In [270]:
def transformer_d8_h4_dim64c(**kwargs: Any) -> TransformerModel:
    num_layers = 1 #Check it on {1,2,3,4,5,6,7}
    num_heads = 4 #Check it on {1,2,3,5,6}
    d_model = 128 #32 {Check on 32,64,128,356,512,768}
    ff_dim = 6*d_model
    embed_mode = "cnn"
    return _transformer("transformer_d8_h4_dim64c", num_layers, num_heads,
                        d_model, ff_dim, embed_mode, **kwargs)

In [271]:
class SpatialAttention1DCBAM(nn.Module):
    def __init__(self, kernel_size=3):
        super(SpatialAttention1DCBAM, self).__init__()
        assert kernel_size % 2 == 1, "Kernel size must be odd"
        
        # 2 → because we concatenate avg + max pool along channel dim
        self.conv1 = nn.Conv1d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        """
        x: Tensor of shape (batch, channels, seq_len)
        """
        # Average pooling along channel axis → (batch, 1, seq_len)
        avg_out = torch.mean(x, dim=1, keepdim=True)
        # Max pooling along channel axis → (batch, 1, seq_len)
        max_out, _ = torch.max(x, dim=1, keepdim=True)

        # Concatenate → (batch, 2, seq_len)
        concat = torch.cat([avg_out, max_out], dim=1)

        # Apply conv + sigmoid → (batch, 1, seq_len)
        attention = self.sigmoid(self.conv1(concat))

        # Multiply with input → (batch, channels, seq_len)
        out = x * (1-attention)
        return out


In [272]:
class TransferModel(nn.Module):
    def __init__(self, encoder_name, n_classes, **kwargs):
        super(TransferModel, self).__init__()
        
        self.encoder_name = encoder_name
        self.encoder = transformer_d8_h4_dim64c(**kwargs)
        print("Loaded {} backbone.".format(self.encoder_name))

        # if freeze_encoder:                                    # For Linear Probing : freeze encoder = True , else False
        #     for param in self.encoder.parameters():
        #         param.requires_grad = False
        
        feat_in = self.encoder.embedding_dim
        # self.classifier = nn.Linear(feat_in, n_classes)
        # Redefine the classifier as a sequence of layers
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),  # Dropout layer added before the final linear layer
            nn.Linear(feat_in, n_classes)
        )

    def forward(self, x):
        feats = self.encoder(x)
        logits = self.classifier(feats)
        return {"logits": logits}

In [273]:
import numpy as np
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    confusion_matrix, roc_auc_score
)
import warnings
from sklearn.exceptions import UndefinedMetricWarning

# Optional: globally suppress undefined metric warnings
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

def safe_roc_auc(y_true, y_score):
    """
    Compute ROC AUC only if both classes are present in y_true.
    """
    if len(np.unique(y_true)) < 2:
        return None  # Could return 0.0 or np.nan if needed
    return roc_auc_score(y_true, y_score)

def compute_classification_metrics(y_true, y_pred, y_score=None):
    """
    Computes per-class and overall classification metrics.
    
    Args:
        y_true (np.ndarray): Ground truth binary labels of shape (N, C)
        y_pred (np.ndarray): Predicted binary labels of shape (N, C)
        y_score (np.ndarray): Probability or score predictions of shape (N, C), required for AUC

    Returns:
        per_class_metrics (dict): Dict of lists containing per-class metrics
        overall_metrics (dict): Dict of overall micro-averaged metrics
    """
    precisions, recalls, f1_scores, accuracies, specificities, aucs = [], [], [], [], [], []

    for i in range(y_true.shape[1]):
        true_labels = y_true[:, i]
        predicted_labels = y_pred[:, i]
        scores = y_score[:, i] if y_score is not None else None

        precision = precision_score(true_labels, predicted_labels, zero_division=0)
        recall = recall_score(true_labels, predicted_labels, zero_division=0)
        f1 = f1_score(true_labels, predicted_labels, zero_division=0)
        accuracy = accuracy_score(true_labels, predicted_labels)
        
        # Safe confusion matrix
        cm = confusion_matrix(true_labels, predicted_labels, labels=[0, 1])
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        else:
            specificity = 0

        # Safe ROC AUC
        auc = safe_roc_auc(true_labels, scores) if scores is not None else None

        # Append metrics
        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        accuracies.append(accuracy)
        specificities.append(specificity)
        aucs.append(auc)

    # Micro-averaged overall metrics
    overall_precision = precision_score(y_true, y_pred, average='micro', zero_division=0)
    overall_recall = recall_score(y_true, y_pred, average='micro', zero_division=0)
    overall_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    overall_accuracy = accuracy_score(y_true.flatten(), y_pred.flatten())

    # Safe overall AUC (micro)
    overall_auc = None
    if y_score is not None:
        try:
            overall_auc = roc_auc_score(y_true, y_score, average="micro")
        except ValueError:
            overall_auc = None

    return {
        "per_class": {
            "precision": precisions,
            "recall": recalls,
            "f1": f1_scores,
            "accuracy": accuracies,
            "specificity": specificities,
            "auc": aucs
        },
        "overall": {
            "precision": overall_precision,
            "recall": overall_recall,
            "f1": overall_f1,
            "accuracy": overall_accuracy,
            "auc": overall_auc
        }
    }


In [274]:
# def train(model, train_loader, criterion, optimizer, device, epoch, num_epochs, scheduler, threshold=0.3):
#     model.train()
#     running_loss = 0.0
    
#     all_train_preds = []
#     all_train_probs = []
#     all_train_labels = []

#     num_exact_matches = 0
#     total_samples_processed = 0

#     with tqdm(total=len(train_loader), desc=f'Epoch {epoch+1}/{num_epochs}', unit='batch') as tepoch:
#         for seq_data, labels in train_loader:
#             seq_data, labels = seq_data.to(device), labels.float().to(device)

#             optimizer.zero_grad()
#             outputs = model(seq_data)
#             loss = criterion(outputs["logits"], labels)
#             if torch.isnan(loss) or torch.isinf(loss):
#                 print("❌ NaN or Inf in loss detected. Skipping this batch.")
#                 print("Logits sample:", outputs["logits"][0])
#                 print("Labels sample:", labels[0])
#                 continue
#             loss.backward()
#             torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#             optimizer.step()

#             running_loss += loss.item()

#             # Probabilities and predictions
#             probs = torch.sigmoid(outputs["logits"])
#             preds = (probs > threshold).int()

#             # Store for epoch-level metrics
#             all_train_preds.extend(preds.detach().cpu().numpy())
#             all_train_probs.extend(probs.detach().cpu().numpy())
#             all_train_labels.extend(labels.detach().cpu().numpy())

#             # Exact match accuracy (batch-level)
#             batch_exact_matches = (preds == labels.int()).all(dim=1).sum().item()
#             num_exact_matches += batch_exact_matches
#             total_samples_processed += labels.size(0)

#             tepoch.set_postfix(
#                 loss=running_loss / (tepoch.n + 1),
#                 exact_match_acc=f"{(num_exact_matches / total_samples_processed) * 100:.2f}%"
#             )
#             tepoch.update(1)

#     scheduler.step()
#     current_lr = scheduler.get_last_lr()[0]
#     print(f"Learning Rate: {current_lr:.6f}")

#     # Convert collected outputs to numpy arrays
#     all_train_preds = np.array(all_train_preds)
#     all_train_probs = np.array(all_train_probs)
#     all_train_labels = np.array(all_train_labels)

#     # === Metrics using shared function ===
#     metrics = compute_classification_metrics(all_train_labels, all_train_preds, all_train_probs)

#     # AUC calculation (use safe handling)
#     train_auc = metrics["overall"]["auc"]

#     # # AUC calculation
#     # try:
#     #     if all_train_labels.shape[1] > 1:
#     #         train_auc = roc_auc_score(all_train_labels, all_train_probs, average='weighted', multi_class='ovr')
#     #     else:
#     #         train_auc = roc_auc_score(all_train_labels, all_train_probs)
#     # except ValueError as e:
#     #     print(f"Warning: Could not compute AUC. {e}")
#     #     train_auc = float('nan')

#     return {
#         "loss": running_loss / len(train_loader),
#         "exact_match_accuracy": (num_exact_matches / total_samples_processed),
#         "precision": metrics["overall"]["precision"],
#         "recall": metrics["overall"]["recall"],
#         "f1_score": metrics["overall"]["f1"],
#         "accuracy": metrics["overall"]["accuracy"] ,
#         "auc": train_auc
#     }


In [275]:
def compute_classification_metrics_multiclass(y_true, y_pred):
    """
    Computes overall classification metrics for a multi-class problem.
    """
    overall_accuracy = accuracy_score(y_true, y_pred)
    overall_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    overall_precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    overall_recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)

    return {
        "overall_accuracy": overall_accuracy,
        "overall_f1": overall_f1,
        "overall_precision": overall_precision,
        "overall_recall": overall_recall,
    }

In [276]:
def train(model, train_loader, criterion, optimizer, device, epoch, num_epochs, scheduler):
    model.train()
    running_loss = 0.0
    
    all_train_preds = []
    all_train_labels = []

    with tqdm(total=len(train_loader), desc=f'Epoch {epoch+1}/{num_epochs}', unit='batch') as tepoch:
        for seq_data, labels in train_loader:
            seq_data, labels = seq_data.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(seq_data)
            logits = outputs["logits"]
            
            loss = criterion(logits, labels)
            if torch.isnan(loss) or torch.isinf(loss):
                print("❌ NaN or Inf in loss detected. Skipping this batch.")
                continue
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            all_train_preds.extend(preds.detach().cpu().numpy())
            all_train_labels.extend(labels.detach().cpu().numpy())

            tepoch.set_postfix(
                loss=running_loss / (tepoch.n + 1),
                acc=f"{(preds == labels).float().mean() * 100:.2f}%"
            )
            tepoch.update(1)

    # scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    print(f"Learning Rate: {current_lr:.6f}")

    all_train_preds = np.array(all_train_preds)
    all_train_labels = np.array(all_train_labels)
    
    metrics = compute_classification_metrics_multiclass(all_train_labels, all_train_preds)

    return {
        "loss": running_loss / len(train_loader),
        "accuracy": metrics["overall_accuracy"],
        "f1_score": metrics["overall_f1"],
        "precision": metrics["overall_precision"],
        "recall": metrics["overall_recall"],
    }

In [277]:
# def evaluate(model, val_loader, criterion, device,epoch, num_epochs, threshold):
#     model.eval()
#     running_loss = 0.0

#     all_val_preds = []
#     all_val_probs = []
#     all_val_labels = []

#     num_exact_matches = 0
#     total_samples_processed = 0

#     with torch.no_grad():
#         for seq_data, labels in val_loader:
#             seq_data, labels = seq_data.to(device), labels.float().to(device)

#             outputs = model(seq_data)
#             loss = criterion(outputs["logits"], labels)
#             running_loss += loss.item()

#             probs = torch.sigmoid(outputs["logits"])
#             preds = (probs > threshold).int()

#             # Store for metrics
#             all_val_preds.extend(preds.detach().cpu().numpy())
#             all_val_probs.extend(probs.detach().cpu().numpy())
#             all_val_labels.extend(labels.detach().cpu().numpy())

#             # Exact match (multi-label setting)
#             batch_exact_matches = (preds == labels.int()).all(dim=1).sum().item()
#             num_exact_matches += batch_exact_matches
#             total_samples_processed += labels.size(0)

#     # Convert to arrays
#     all_val_preds = np.array(all_val_preds)
#     all_val_probs = np.array(all_val_probs)
#     all_val_labels = np.array(all_val_labels)

#     # === Metrics ===
#     metrics = compute_classification_metrics(all_val_labels, all_val_preds, all_val_probs)
    
#     # AUC calculation (use safe handling)
#     val_auc = metrics["overall"]["auc"]

#     # # AUC calculation
#     # try:
#     #     if all_val_labels.shape[1] > 1:
#     #         val_auc = roc_auc_score(all_val_labels, all_val_probs, average='weighted', multi_class='ovr')
#     #     else:
#     #         val_auc = roc_auc_score(all_val_labels, all_val_probs)
#     # except ValueError as e:
#     #     print(f"Warning: Could not compute AUC. {e}")
#     #     val_auc = float('nan')

#     return {
#         "loss": running_loss / len(val_loader),
#         "exact_match_accuracy": (num_exact_matches / total_samples_processed) ,
#         "overall_precision": metrics["overall"]["precision"],
#         "overall_recall": metrics["overall"]["recall"],
#         "overall_f1": metrics["overall"]["f1"],
#         "overall_accuracy": metrics["overall"]["accuracy"],
#         "auc": val_auc if val_auc is not None else 0.0   # avoid nan
#     }
    


In [278]:
# def evaluate(model, val_loader, criterion, device, epoch, num_epochs):
#     model.eval()
#     running_loss = 0.0

#     all_val_preds = []
#     all_val_labels = []

#     with torch.no_grad():
#         for seq_data, labels in val_loader:
#             seq_data, labels = seq_data.to(device), labels.to(device)

#             outputs = model(seq_data)
#             logits = outputs["logits"]
#             loss = criterion(logits, labels)
#             running_loss += loss.item()

#             preds = torch.argmax(logits, dim=1)

#             all_val_preds.extend(preds.detach().cpu().numpy())
#             all_val_labels.extend(labels.detach().cpu().numpy())

#     all_val_preds = np.array(all_val_preds)
#     all_val_labels = np.array(all_val_labels)

#     metrics = compute_classification_metrics_multiclass(all_val_labels, all_val_preds)
    
#     return {
#         "loss": running_loss / len(val_loader),
#         "accuracy": metrics["overall_accuracy"],
#         "f1_score": metrics["overall_f1"],
#         "precision": metrics["overall_precision"],
#         "recall": metrics["overall_recall"],
#     }

In [279]:
from sklearn.metrics import roc_auc_score
import torch
import numpy as np

def evaluate(model, val_loader, criterion, device, epoch, num_epochs):
    model.eval()
    running_loss = 0.0

    all_val_preds = []
    all_val_labels = []
    all_val_probs = []   # <-- for AUROC

    with torch.no_grad():
        for seq_data, labels in val_loader:
            seq_data, labels = seq_data.to(device), labels.to(device)

            outputs = model(seq_data)
            logits = outputs["logits"]

            loss = criterion(logits, labels)
            running_loss += loss.item()

            probs = torch.softmax(logits, dim=1)     # <-- probabilities
            preds = torch.argmax(probs, dim=1)

            all_val_preds.append(preds.cpu().numpy())
            all_val_labels.append(labels.cpu().numpy())
            all_val_probs.append(probs.cpu().numpy())

    all_val_preds = np.concatenate(all_val_preds)
    all_val_labels = np.concatenate(all_val_labels)
    all_val_probs = np.concatenate(all_val_probs)

    metrics = compute_classification_metrics_multiclass(
        all_val_labels, all_val_preds
    )

    # ---------- AUROC (multiclass) ----------
    try:
        auroc = roc_auc_score(
            all_val_labels,
            all_val_probs,
            multi_class="ovr",   # one-vs-rest (recommended)
            average="macro"
        )
    except ValueError:
        auroc = float("nan")  # happens if a class is missing in val set

    return {
        "loss": running_loss / len(val_loader),
        "accuracy": metrics["overall_accuracy"],
        "f1_score": metrics["overall_f1"],
        "precision": metrics["overall_precision"],
        "recall": metrics["overall_recall"],
        "auroc": auroc,   # <-- added
    }


In [280]:
# train_path = "/home/durgesh/Sandeep_11_10_Ajeet_Foundation/Normalized_Data/cpsc_xlNormalized/train_normalized.npy"
# label_path = "/home/durgesh/Sandeep_11_10_Ajeet_Foundation/Normalized_Data/cpsc_xlNormalized/y_train.npy"
# val_train_path = "/home/durgesh/Sandeep_11_10_Ajeet_Foundation/Normalized_Data/cpsc_xlNormalized/val_normalized.npy"
# val_label_path = "/home/durgesh/Sandeep_11_10_Ajeet_Foundation/Normalized_Data/cpsc_xlNormalized/y_val.npy"
# eval_train_path = "/home/durgesh/Sandeep_11_10_Ajeet_Foundation/Normalized_Data/cpsc_xlNormalized/test_normalized.npy"
# eval_label_path = "/home/durgesh/Sandeep_11_10_Ajeet_Foundation/Normalized_Data/cpsc_xlNormalized/y_test.npy"

In [281]:
train_path = "/home/durgesh/Sandeep_11_10_Ajeet_Foundation/New DataFiles/PTB-XL/X_train.npy"
label_path = "/home/durgesh/Sandeep_11_10_Ajeet_Foundation/New_Nomralized_data/Ptb_XL_Normalized/y_train_cleaned.npy"
val_train_path = "/home/durgesh/Sandeep_11_10_Ajeet_Foundation/New DataFiles/PTB-XL/X_val.npy"
val_label_path = "/home/durgesh/Sandeep_11_10_Ajeet_Foundation/New_Nomralized_data/Ptb_XL_Normalized/y_val_cleaned.npy"
eval_train_path = "/home/durgesh/Sandeep_11_10_Ajeet_Foundation/New DataFiles/PTB-XL/X_test.npy"
eval_label_path = "/home/durgesh/Sandeep_11_10_Ajeet_Foundation/New_Nomralized_data/Ptb_XL_Normalized/y_test_cleaned.npy"

In [282]:
train_data = train_DatasetWrapper(train_path, label_path)
val_data = train_DatasetWrapper(val_train_path, val_label_path)  # No augmentation for validation
eval_data = train_DatasetWrapper(eval_train_path, eval_label_path)  # No augmentation for test

In [283]:
batch_size = 512      # 512
num_workers = 4
lr = 1e-6             # 0.0001
weight_decay = 0.05      # 0.01
embedding_dim = 256     # 512
encoder_name = "transformer"
output_dim = 256
proj_hidden_dim = 2048
temperature = 0.07
num_epochs = 150
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

In [284]:
num_classes = 5
num_epochs = 100

In [285]:
train_loader = DataLoader(train_data, batch_size=batch_size,  num_workers=num_workers, shuffle=True)
val_loader = DataLoader(val_data, batch_size = batch_size, num_workers = num_workers, shuffle = False)
eval_loader = DataLoader(eval_data, batch_size = batch_size, num_workers = num_workers, shuffle = False)

In [286]:
model = TransferModel(encoder_name,n_classes= num_classes, embedding_dim=embedding_dim)

Transformer with embedding dim 256
Loaded transformer backbone.


In [287]:
model = model.to(device)
# # optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9,0.999))
# optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

In [288]:
## Load the pre-trained model
print("Loading pre-trained model...")
# state_dict = torch.load("/home/tarun/Documents/IR Spectra Classification Reasearch/Overall Pretrained Models/pretrained_models3/17_9_25_tanishk/model_epoch_130.pth", map_location='cpu',weights_only=True)
state_dict = torch.load("/home/durgesh/Sandeep_11_10_Ajeet_Foundation/Pretrained_Model/23_10_25_pt/best_model.pt", map_location='cpu',weights_only=True)

Loading pre-trained model...


In [289]:
# Filter only encoder-related keys and remove prefix
from collections import OrderedDict

encoder_state_dict = OrderedDict()
# for k, v in state_dict.items():
#     if k.startswith("encoder."):  # or "encoder." if that's the prefix
#         new_key = k.replace("encoder.", "")  # remove the prefix to match current encoder
#         encoder_state_dict[new_key] = v
        
for k, v in state_dict.items():
    if k.startswith("base_model.encoder."):  # or "encoder." if that's the prefix
        new_key = k.replace("base_model.encoder.", "")  # remove the prefix to match current encoder
        encoder_state_dict[new_key] = v
print(f"Found {len(encoder_state_dict)} encoder layers to load")

Found 32 encoder layers to load


In [290]:
# Print the names of the encoder layers being loaded
print("Encoder layer names loaded from state_dict:")
for name in encoder_state_dict.keys():
    print(name)

Encoder layer names loaded from state_dict:
embed.conv1.weight
embed.conv1.bias
embed.conv2.weight
embed.conv2.bias
embed.conv3.weight
embed.conv3.bias
embed.conv6.weight
embed.conv6.bias
embed.attn6.conv1.weight
embed.attn6.conv1.bias
embed.dense.weight
embed.dense.bias
pos_encoder.pe
transformer_encoder.0.norm1.weight
transformer_encoder.0.norm1.bias
transformer_encoder.0.attn.m_t
transformer_encoder.0.attn.m_c
transformer_encoder.0.attn.query.weight
transformer_encoder.0.attn.key.weight
transformer_encoder.0.attn.content_bias.weight
transformer_encoder.0.attn.value.weight
transformer_encoder.0.attn.value.bias
transformer_encoder.0.attn.dense.weight
transformer_encoder.0.attn.dense.bias
transformer_encoder.0.norm2.weight
transformer_encoder.0.norm2.bias
transformer_encoder.0.ff.0.weight
transformer_encoder.0.ff.0.bias
transformer_encoder.0.ff.3.weight
transformer_encoder.0.ff.3.bias
fc.weight
fc.bias


In [291]:
for param in model.encoder.parameters():
        param.requires_grad = True # For Fine Tuning
    
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)                   
criterion = nn.CrossEntropyLoss()
# criterion = FocalLoss(alpha=0.05, gamma=1.0)

In [292]:
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-8)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-8)

In [293]:
save_path = "/home/durgesh/Sandeep_11_10_Ajeet_Foundation/finetunig_model/chap,an_linearProb"
# save_path = "/home/durgesh/ecg_fm/Classification_IR_Spectra/LinearProbed Models/3K"
os.makedirs(save_path, exist_ok=True)

In [294]:
best_f1 = 0.0
best_acc = 0.0
best_pre = 0.0
best_rec = 0.0
best_model_state = None
patience_counter = 0
patience = 20

In [295]:
# for epoch in range(num_epochs):
#     # === Train ===
#     train_metrics = train(model, train_loader, criterion, optimizer, device, epoch, num_epochs, scheduler)

#     # === Evaluate ===
#     val_metrics = evaluate(model, val_loader, criterion, device, epoch, num_epochs)
    
#     torch.cuda.empty_cache()

#     print(f"\nEpoch {epoch+1}/{num_epochs}")
#     print(f"Train  - Loss: {train_metrics['loss']:.4f}, Acc: {train_metrics['accuracy'] * 100:.2f}%, F1: {train_metrics['f1_score']:.4f}, Precision: {train_metrics['precision']:.4f}, Recall: {train_metrics['recall']:.4f}")
#     print(f"Val    - Loss: {val_metrics['loss']:.4f}, Acc: {val_metrics['accuracy'] * 100:.2f}%, F1: {val_metrics['f1_score']:.4f}, Precision: {val_metrics['precision']:.4f}, Recall: {val_metrics['recall']:.4f}")
    
#     # === Save if current F1 is better ===
#     if val_metrics["accuracy"] > best_acc:
#         best_f1 = val_metrics["f1_score"]
#         best_acc = val_metrics["accuracy"]
#         best_model_state = copy.deepcopy(model.state_dict())
#         patience_counter = 0

#         model_file = os.path.join(
#             save_path,
#             f"best_model_epoch(f1={best_f1:.4f}_acc={best_acc:.2f}).pth"
#         )
#         torch.save(best_model_state, model_file)
#         print(f"✅ New best model saved to: {model_file}")
#     else:
#         patience_counter += 1
#         print(f"❌ No improvement for {patience_counter} epochs")

#     # === Early stopping ===
#     if patience_counter >= patience:
#         print(f"🛑 Early stopping triggered after {patience} epochs without improvement")
#         break

#     torch.cuda.empty_cache()

# # === Summary ===
# print(f"\n=== Training Complete ===")
# print(f"Best validation accuracy: {best_acc * 100:.2f}%")
# print(f"Best validation F1 score: {best_f1:.4f}")

In [ ]:
best_acc = 0.0
best_f1 = 0.0
best_auroc = 0.0
patience_counter = 0

for epoch in range(num_epochs):
    # === Train ===
    train_metrics = train(
        model, train_loader, criterion, optimizer,
        device, epoch, num_epochs, scheduler
    )

    # === Evaluate ===
    val_metrics = evaluate(
        model, val_loader, criterion, device, epoch, num_epochs
    )

    torch.cuda.empty_cache()

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print(
        f"Train  - "
        f"Loss: {train_metrics['loss']:.4f}, "
        f"Acc: {train_metrics['accuracy'] * 100:.2f}%, "
        f"F1: {train_metrics['f1_score']:.4f}, "
        f"Precision: {train_metrics['precision']:.4f}, "
        f"Recall: {train_metrics['recall']:.4f}"
    )
    print(
        f"Val    - "
        f"Loss: {val_metrics['loss']:.4f}, "
        f"Acc: {val_metrics['accuracy'] * 100:.2f}%, "
        f"F1: {val_metrics['f1_score']:.4f}, "
        f"Precision: {val_metrics['precision']:.4f}, "
        f"Recall: {val_metrics['recall']:.4f}, "
        f"AUROC: {val_metrics['auroc']:.4f}"
    )

    # === Save if current accuracy is better ===
    if val_metrics["auroc"] > best_auroc:

        best_acc = val_metrics["accuracy"]
        best_f1 = val_metrics["f1_score"]
        best_auroc = val_metrics["auroc"]

        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0

        model_file = os.path.join(
            save_path,
            f"best_model_epoch("
            f"acc={best_acc:.4f}_"
            f"f1={best_f1:.4f}_"
            f"auroc={best_auroc:.4f}"
            f").pth"
        )

        torch.save(best_model_state, model_file)
        print(f"✅ New best model saved to: {model_file}")

    else:
        patience_counter += 1
        print(f"❌ No improvement for {patience_counter} epochs")

    # === Early stopping ===
    if patience_counter >= patience:
        print(f"🛑 Early stopping triggered after {patience} epochs without improvement")
        break

    torch.cuda.empty_cache()

# === Summary ===
print(f"\n=== Training Complete ===")
print(f"Best validation accuracy: {best_acc * 100:.2f}%")
print(f"Best validation F1 score: {best_f1:.4f}")
print(f"Best validation AUROC: {best_auroc:.4f}")


Epoch 1/100: 100%|██████████| 34/34 [00:09<00:00,  3.45batch/s, acc=5.85%, loss=1.65]


Learning Rate: 0.000001

Epoch 1/100
Train  - Loss: 1.6459, Acc: 5.63%, F1: 0.0134, Precision: 0.0078, Recall: 0.0563
Val    - Loss: 1.6424, Acc: 4.80%, F1: 0.0044, Precision: 0.0023, Recall: 0.0480, AUROC: 0.4964
✅ New best model saved to: /home/durgesh/Sandeep_11_10_Ajeet_Foundation/finetunig_model/chap,an_linearProb/best_model_epoch(acc=0.0480_f1=0.0044_auroc=0.4964).pth


Epoch 2/100: 100%|██████████| 34/34 [00:09<00:00,  3.43batch/s, acc=7.45%, loss=1.64]

Learning Rate: 0.000001



Epoch 2/100
Train  - Loss: 1.6387, Acc: 6.13%, F1: 0.0141, Precision: 0.2295, Recall: 0.0613
Val    - Loss: 1.6347, Acc: 7.92%, F1: 0.0117, Precision: 0.0063, Recall: 0.0792, AUROC: 0.4968
✅ New best model saved to: /home/durgesh/Sandeep_11_10_Ajeet_Foundation/finetunig_model/chap,an_linearProb/best_model_epoch(acc=0.0792_f1=0.0117_auroc=0.4968).pth


...


Epoch 64/100
Train  - Loss: 1.3524, Acc: 44.31%, F1: 0.2721, Precision: 0.1963, Recall: 0.4431
Val    - Loss: 1.3769, Acc: 44.22%, F1: 0.2712, Precision: 0.1956, Recall: 0.4422, AUROC: 0.5054
❌ No improvement for 4 epochs


Epoch 65/100: 100%|██████████| 34/34 [00:10<00:00,  3.24batch/s, acc=54.26%, loss=1.35]

Learning Rate: 0.000001



Epoch 65/100
Train  - Loss: 1.3532, Acc: 44.30%, F1: 0.2721, Precision: 0.1963, Recall: 0.4430
Val    - Loss: 1.3769, Acc: 44.22%, F1: 0.2712, Precision: 0.1956, Recall: 0.4422, AUROC: 0.5058
✅ New best model saved to: /home/durgesh/Sandeep_11_10_Ajeet_Foundation/finetunig_model/chap,an_linearProb/best_model_epoch(acc=0.4422_f1=0.2712_auroc=0.5058).pth


Epoch 66/100:   0%|          | 0/34 [00:00<?, ?batch/s]